# 02 — Main Analysis

End-to-end analysis pipeline: embeddings → consensus UMAP → clustering → projection → salience ratios.

Uses `src/` modules for all computation. See `03_consensus_umap_details.ipynb` for methodology deep-dive.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loading import load_public_discourse, load_artist_perspectives, load_likert_anchors
from src.embeddings import load_or_embed, embed_chunks
from src.consensus_umap import (
    run_umap_multi_seed, distance_matrix_consensus,
    umap_from_precomputed_distances, procrustes_align, compute_consensus_average
)
from src.clustering import run_kmeans, run_hdbscan, find_best_k, get_cluster_sizes, top_ngrams
from src.projection import train_projection_head, project_to_consensus_space
from src.analysis import (
    safe_probabilities, js_divergence, topic_counts,
    centroid_distance, compute_salience_ratios, cramers_v
)
from src.visualization import plot_clusters_2d, plot_artist_concentration, plot_salience_ratios
from src.models import PipelineConfig

## 1. Load Data

In [ ]:
config = PipelineConfig()
df_public = load_public_discourse('../data/')
df_artist = load_artist_perspectives('../data/')
df_likert = load_likert_anchors('../data/')
print(f'Public: {len(df_public)} chunks | Artists: {len(df_artist)} probes | Likert: {len(df_likert)} anchors')

## 2. Generate or Load Embeddings

Using `intfloat/e5-large-v2` (1024-dim). Set `precomputed_npy` to skip re-encoding.

In [ ]:
# Set to a .npy path if you have precomputed embeddings, else None to compute
PRECOMPUTED_PUBLIC = None  # e.g., '../embeddings/public_e5.npy'
PRECOMPUTED_ARTIST = None  # e.g., '../embeddings/artist_e5.npy'

X_public = load_or_embed(df_public, 'chunk_text_clean', config.embedding.model_name, PRECOMPUTED_PUBLIC)
X_artist = load_or_embed(df_artist, 'perspective_text', config.embedding.model_name, PRECOMPUTED_ARTIST)
print(f'Public embeddings: {X_public.shape} | Artist embeddings: {X_artist.shape}')

## 3. Consensus UMAP

31 seeds → pairwise distance matrices → averaged → final 8D embedding.

In [ ]:
from src.consensus_umap import l2_normalize

X_norm = l2_normalize(X_public)
umap_embeddings = run_umap_multi_seed(
    X_norm, seeds=config.umap.seeds,
    n_components=config.umap.n_components,
    n_neighbors=config.umap.n_neighbors,
    min_dist=config.umap.min_dist,
    metric=config.umap.metric,
)
print(f'Generated {len(umap_embeddings)} UMAP embeddings, each shape {umap_embeddings[0].shape}')

In [ ]:
# Distance-matrix consensus (recommended: ARI 0.71 vs 0.56 for coordinate averaging)
D_consensus = distance_matrix_consensus(umap_embeddings, metric='euclidean')
print(f'Consensus distance matrix: {D_consensus.shape}')

# Final UMAP from precomputed distances
consensus_coords, _ = umap_from_precomputed_distances(
    D_consensus, n_components=config.umap.n_components,
    n_neighbors=config.umap.n_neighbors, min_dist=config.umap.min_dist,
)
print(f'Consensus coordinates: {consensus_coords.shape}')

## 4. Clustering

In [ ]:
labels, clusterer = run_hdbscan(consensus_coords, min_cluster_size=10, min_samples=5)
sizes = get_cluster_sizes(labels)
n_clusters = len([k for k in sizes if k >= 0])
n_noise = sizes.get(-1, 0)
print(f'Clusters: {n_clusters} | Noise points: {n_noise} ({100*n_noise/len(labels):.1f}%)')

## 5. Projection Head + Cross-Corpus Comparison

Train MLP to map embeddings → consensus space, then project artist probes.

In [ ]:
result = train_projection_head(
    X_public, consensus_coords,
    hidden_layer_sizes=config.projection.hidden_layer_sizes,
    random_state=config.projection.random_state,
)
print(f'Projection head R² train: {result["r2_train"]:.4f} | val: {result["r2_val"]:.4f}')

# Project artist probes
artist_projected = project_to_consensus_space(
    X_artist, result['model'], result['scaler_X'], result['scaler_Y']
)
print(f'Artist probes projected: {artist_projected.shape}')

## 6. Salience Analysis

In [ ]:
# Assign artist probes to nearest cluster
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=1).fit(consensus_coords)
_, idx = nn.kneighbors(artist_projected)
artist_labels = labels[idx.flatten()]

# Compute topic distributions
all_labels_list = sorted([l for l in set(labels) if l >= 0])
public_counts = topic_counts(pd.DataFrame({'cluster': labels}), 'cluster', all_labels_list)
artist_counts = topic_counts(pd.DataFrame({'cluster': artist_labels}), 'cluster', all_labels_list)

# Salience ratios
ratios = compute_salience_ratios(artist_counts, public_counts, all_labels_list)
for r in sorted(ratios, key=lambda x: x['salience_ratio'], reverse=True)[:5]:
    print(f"Topic {r['topic_label']}: SR={r['salience_ratio']:.2f} "
          f"(artist={r['artist_frac']:.3f}, public={r['public_frac']:.3f})")